# Chapter 29
## Stability of the Synchronous State
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter29.ipynb)

## About this chapter

Synchrony is stable when an inhibitory perturbation contracts a small timing
difference between two nearly-synchronous cells rather than amplifying it.
These examples calculate the maps that carry a pre-pulse timing difference to
a post-pulse one -- for LIF cells (whose hard reset makes the map easy to
compute) and for conductance-based RTM cells -- and check how sensitive the
inferred stability is to small changes in parameters. A geometric
theta-neuron "river" picture closes the chapter by showing the same
contraction/expansion as a flow in the $(\theta,g_{\rm syn})$ plane.

Two nearby phases are represented by their pre-pulse and post-pulse
pulse-to-spike timings $P_0$ and $P_1$. The LIF examples define their mean
timing and normalized separation as

$$
P=\frac{P_0+P_1}{2},\qquad S=\frac{P_0-P_1}{P}.
$$

$S$, rather than a derivative of $P$, is the plotted stability quantity:
small $S$ means the two timings have nearly merged, i.e. synchrony is
locally stable. The condition-number calculations report how much $P$
changes (in percent) under a 1% change in a parameter -- a large percentage
change means the conclusion is numerically fragile even if the sign of $S$
says synchrony is stable.

See [`chapter29.md`](chapter29.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from ipywidgets import interact
from mnd.core import draw_arrow

## Illustrating $P_0$ and $P_1$

Two LIF trajectories under the same decaying inhibitory conductance
$g\,e^{-t/\tau_I}$: one starting from $v(0)=0$ (giving $P_0$, the
pulse-to-spike time of a cell reset by its own spike) and one from $v(0)=1$
(giving $P_1$, the timing of a cell that was already at threshold when the
pulse arrived). `simulate_p0_and_p1` Heun-integrates both until threshold
and returns each trace with its interpolated threshold-crossing time.

In [ ]:
def simulate_p0_and_p1(tau_m=10., I=0.12, g=0.15, tau_I=7., dt=0.01):
    '''Heun-integrate a LIF neuron receiving a decaying inhibitory
    conductance g*exp(-t/tau_I) starting at t=0, once from v(0)=0 (P0) and
    once from v(0)=1 (P1), until it reaches threshold v=1. Returns each
    run\'s voltage trace, step count, and interpolated threshold-crossing
    time ("period").'''
    dt05 = dt / 2

    def run(v0, continue_while):
        v = [v0]
        k = 0
        while continue_while(v[k]):
            t = k * dt
            v_inc = -v[k] / tau_m + I - g * np.exp(-t / tau_I) * v[k]
            v_tmp = v[k] + dt05 * v_inc
            v_inc = -v_tmp / tau_m + I - g * np.exp(-(t + dt05) / tau_I) * v_tmp
            v.append(v[k] + dt * v_inc)
            k += 1
        v = np.array(v)
        period = ((k - 1) * dt * (v[k] - 1) + k * dt * (1 - v[k - 1])) / (v[k] - v[k - 1])
        return v, k, period

    v_0, k_0, period_0 = run(0., lambda vk: vk < 1)
    v_1, k_1, period_1 = run(1., lambda vk: vk <= 1)
    return v_0, k_0, period_0, v_1, k_1, period_1


def plot_p0_and_p1(v_0, k_0, period_0, v_1, k_1, period_1, dt=0.01):
    plt.figure(figsize=(8, 5))

    t_0 = np.arange(k_0) * dt
    plt.plot(t_0, v_0[:k_0], '-k', linewidth=2)
    plt.plot([(k_0 - 1) * dt, period_0], [v_0[k_0 - 1], 1], '-k', linewidth=2)
    plt.plot([period_0, period_0], [0, 1], ':k', linewidth=2)
    plt.text(period_0 - 0.5, -0.15, '$P_0$', fontsize=16)

    t_1 = np.arange(k_1) * dt
    plt.plot(t_1, v_1[:k_1], '-k', linewidth=2)
    plt.plot([(k_1 - 1) * dt, period_1], [v_1[k_1 - 1], 1], '-k', linewidth=2)
    plt.plot([period_1, period_1], [0, 1], ':k', linewidth=2)
    plt.text(period_1 - 1, -0.15, '$P_1$', fontsize=16)

    plt.xticks(range(0, 31, 15))
    plt.axis([0, 30, 0, 1])
    plt.tight_layout()
    plt.show()

In [ ]:
v_0, k_0, period_0, v_1, k_1, period_1 = simulate_p0_and_p1()
plot_p0_and_p1(v_0, k_0, period_0, v_1, k_1, period_1)

In [ ]:
interact(lambda g=0.15: plot_p0_and_p1(*simulate_p0_and_p1(g=g)),
         g=(0.0, 0.4, 0.01));

## LIF Timing Maps $P_0$, $P_1$, $P$, $S$ (shared by the LIF examples below)

`P0` and `P1` Heun-integrate a single LIF cell under a decaying inhibitory
conductance from $v(0)=0$ and $v(0)=1$ respectively (as above, but as a
function of the model parameters); `P` is their mean and `S` the normalized
separation used as the stability indicator.

In [ ]:
def P0(tau_m, J, g, tau_I, dt=0.01):
    dt05 = dt / 2
    I = J + 1 / tau_m
    v = 0.
    k = 1
    while v < 1:
        v_old = v
        v_inc = -v / tau_m + I - g * np.exp(-(k - 1) * dt / tau_I) * v
        v_tmp = v + dt05 * v_inc
        v_inc = -v_tmp / tau_m + I - g * np.exp(-(k - 0.5) * dt / tau_I) * v_tmp
        v = v + dt * v_inc
        k += 1
    period = ((k - 2) * dt * (v - 1) + (k - 1) * dt * (1 - v_old)) / (v - v_old)
    return period


def P1(tau_m, J, g, tau_I, dt=0.01):
    dt05 = dt / 2
    I = J + 1 / tau_m
    if g <= J:
        return 0.
    v = 1.
    k = 1
    while v <= 1:
        v_old = v
        v_inc = -v / tau_m + I - g * np.exp(-(k - 1) * dt / tau_I) * v
        v_tmp = v + dt05 * v_inc
        v_inc = -v_tmp / tau_m + I - g * np.exp(-(k - 0.5) * dt / tau_I) * v_tmp
        v = v + dt * v_inc
        k += 1
    period = ((k - 2) * dt * (v - 1) + (k - 1) * dt * (1 - v_old)) / (v - v_old)
    return period


def P(tau_m, J, g, tau_I, dt=0.01):
    return (P0(tau_m, J, g, tau_I, dt) + P1(tau_m, J, g, tau_I, dt)) / 2


def S(tau_m, J, g, tau_I, dt=0.01):
    return (P0(tau_m, J, g, tau_I, dt) - P1(tau_m, J, g, tau_I, dt)) / P(tau_m, J, g, tau_I, dt)

## LIF $P$ and $S$

$P$ and $S$ plotted while independently varying $\tau_I$, $\overline{g}_{\rm
syn}$, and $J$ around a baseline operating point, each over a $\pm20\%$
range.

In [ ]:
def simulate_lif_p_and_s(tau_m=10., J_0=0.02, g_0=0.15, tau_I_0=9.):
    tau_I_vec = tau_I_0 * 0.8 + np.arange(101) / 100 * tau_I_0 * 0.4
    g_vec = g_0 * 0.8 + np.arange(101) / 100 * g_0 * 0.4
    J_vec = J_0 * 0.8 + np.arange(101) / 100 * (J_0 * 1.2 - J_0 * 0.8)

    P_vec_tau_I = np.array([P(tau_m, J_0, g_0, tI) for tI in tau_I_vec])
    S_vec_tau_I = np.array([S(tau_m, J_0, g_0, tI) for tI in tau_I_vec])

    P_vec_g = np.array([P(tau_m, J_0, gg, tau_I_0) for gg in g_vec])
    S_vec_g = np.array([S(tau_m, J_0, gg, tau_I_0) for gg in g_vec])

    P_vec_J = np.array([P(tau_m, JJ, g_0, tau_I_0) for JJ in J_vec])
    S_vec_J = np.array([S(tau_m, JJ, g_0, tau_I_0) for JJ in J_vec])

    return (tau_I_vec, P_vec_tau_I, S_vec_tau_I,
            g_vec, P_vec_g, S_vec_g,
            J_vec, P_vec_J, S_vec_J)


def plot_lif_p_and_s(tau_I_vec, P_vec_tau_I, S_vec_tau_I,
                      g_vec, P_vec_g, S_vec_g,
                      J_vec, P_vec_J, S_vec_J,
                      tau_m=10., J_0=0.02, g_0=0.15, tau_I_0=9.):
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))

    axes[0, 0].plot(tau_I_vec, P_vec_tau_I, '-k', linewidth=2)
    axes[0, 0].plot(tau_I_0, P(tau_m, J_0, g_0, tau_I_0), '.r', markersize=15)
    axes[0, 0].axis([tau_I_0 * 0.8, tau_I_0 * 1.2, 20, 40])
    axes[0, 0].set_ylabel('$P$')
    axes[0, 0].set_title(r'varying $\tau_I$')

    axes[1, 0].plot(tau_I_vec, S_vec_tau_I, '-k', linewidth=2)
    axes[1, 0].plot(tau_I_0, S(tau_m, J_0, g_0, tau_I_0), '.r', markersize=15)
    axes[1, 0].axis([tau_I_0 * 0.8, tau_I_0 * 1.2, 0, 0.1])
    axes[1, 0].set_xlabel(r'$\tau_I$')
    axes[1, 0].set_ylabel('$S$')

    axes[0, 1].plot(g_vec, P_vec_g, '-k', linewidth=2)
    axes[0, 1].plot(g_0, P(tau_m, J_0, g_0, tau_I_0), '.r', markersize=15)
    axes[0, 1].set_xticks(np.arange(0.13, 0.171, 0.02))
    axes[0, 1].axis([0.8 * g_0, 1.2 * g_0, 20, 40])
    axes[0, 1].set_title(r'varying $\overline{g}_{\rm syn}$')

    axes[1, 1].plot(g_vec, S_vec_g, '-k', linewidth=2)
    axes[1, 1].plot(g_0, S(tau_m, J_0, g_0, tau_I_0), '.r', markersize=15)
    axes[1, 1].set_xticks(np.arange(0.13, 0.171, 0.02))
    axes[1, 1].set_xlabel(r'$\overline{g}_{\rm syn}$')
    axes[1, 1].axis([0.8 * g_0, 1.2 * g_0, 0, 0.1])

    axes[0, 2].plot(J_vec, P_vec_J, '-k', linewidth=2)
    axes[0, 2].plot(J_0, P(tau_m, J_0, g_0, tau_I_0), '.r', markersize=15)
    axes[0, 2].set_xticks([0.017, 0.02, 0.023])
    axes[0, 2].axis([0.8 * J_0, 1.2 * J_0, 20, 40])
    axes[0, 2].set_title('varying $J$')

    axes[1, 2].plot(J_vec, S_vec_J, '-k', linewidth=2)
    axes[1, 2].plot(J_0, S(tau_m, J_0, g_0, tau_I_0), '.r', markersize=15)
    axes[1, 2].set_xticks([0.017, 0.02, 0.023])
    axes[1, 2].set_xlabel('$J$')
    axes[1, 2].axis([0.8 * J_0, 1.2 * J_0, 0, 0.1])

    plt.tight_layout()
    plt.show()

In [ ]:
(tau_I_vec, P_vec_tau_I, S_vec_tau_I, g_vec, P_vec_g, S_vec_g,
 J_vec, P_vec_J, S_vec_J) = simulate_lif_p_and_s()
plot_lif_p_and_s(tau_I_vec, P_vec_tau_I, S_vec_tau_I, g_vec, P_vec_g, S_vec_g,
                  J_vec, P_vec_J, S_vec_J)

In [ ]:
def _plot_lif_p_and_s(tau_m=10.):
    plot_lif_p_and_s(*simulate_lif_p_and_s(tau_m=tau_m), tau_m=tau_m)


interact(_plot_lif_p_and_s, tau_m=(5.0, 20.0, 0.5));

## LIF Condition Numbers

For a weak/slow synapse (`g=0.15, tau_I=9`) and a strong/fast one
(`g=2, tau_I=1`), `compute_lif_condition_numbers` reports the baseline mean
timing $P$ and the percent change in $P$ under a 1% change in each of $J$,
$g$, and $\tau_I$ -- large percentage changes mean the inferred timing is
numerically sensitive to that parameter.

In [ ]:
def compute_lif_condition_numbers(tau_m=10.):
    results = {}
    for label, (J, g, tau_I) in (
        ('slow', (0.02, 0.15, 9.)),
        ('fast', (0.02, 2., 1.)),
    ):
        P_here = P(tau_m, J, g, tau_I)
        pct_J = (P(tau_m, 0.99 * J, g, tau_I) - P_here) / P_here * 100
        pct_g = (P(tau_m, J, g * 1.01, tau_I) - P_here) / P_here * 100
        pct_tau_I = (P(tau_m, J, g, tau_I * 1.01) - P_here) / P_here * 100
        results[label] = {'P': P_here, 'J': pct_J, 'g': pct_g, 'tau_I': pct_tau_I}
    return results

In [ ]:
results = compute_lif_condition_numbers()
for label, r in results.items():
    print(label, r)

## LIF With Inhibitory Pulse

$N=10$ LIF cells, initialized in the splay state (evenly spread over one
firing cycle), all receive a common decaying inhibitory pulse starting at
$t=0$. `simulate_lif_pulse_panels` runs this for no inhibition, a weak/slow
synapse, and a strong/fast one -- watch how much the cells' firing times
bunch together (or don't) after the pulse in each panel.

In [ ]:
def simulate_lif_pulse_panels(tau_m=10., I=0.12, N=10, dt=0.001):
    T = tau_m * np.log(tau_m * I / (tau_m * I - 1))  # firing period without inhibition
    dt05 = dt / 2

    def simulate_one(v0, g_syn, tau_I):
        v = [v0]
        k = 0
        while v[k] < 1:
            t = k * dt
            v_inc = -v[k] / tau_m + I - g_syn * np.exp(-t / tau_I) * v[k]
            v_tmp = v[k] + dt05 * v_inc
            v_inc = -v_tmp / tau_m + I - g_syn * np.exp(-(t + dt05) / tau_I) * v_tmp
            v.append(v[k] + dt * v_inc)
            k += 1
        v = np.array(v)
        period = ((k - 1) * dt * (v[k] - 1) + k * dt * (1 - v[k - 1])) / (v[k] - v[k - 1])
        return v, k, period

    def simulate_panel(g_syn, tau_I):
        traces = []
        for i in range(1, N + 1):
            v0 = (1 - np.exp(-(N - i) * T / (N * tau_m))) * tau_m * I
            traces.append(simulate_one(v0, g_syn, tau_I))
        return traces

    traces_none = simulate_panel(g_syn=0., tau_I=1.)  # g_syn=0 -> no inhibition
    traces_weak = simulate_panel(g_syn=0.15, tau_I=9.)
    traces_strong = simulate_panel(g_syn=2., tau_I=1.)
    return traces_none, traces_weak, traces_strong


def plot_lif_pulse_panels(traces_none, traces_weak, traces_strong, dt=0.001):
    fig, axes = plt.subplots(3, 1, figsize=(8, 9))

    for ax, traces in zip(axes, (traces_none, traces_weak, traces_strong)):
        for v, k, period in traces:
            t = np.arange(k) * dt
            ax.plot(t, v[:k], '-k', linewidth=1)
            ax.plot([(k - 1) * dt, period], [v[k - 1], 1], '-k', linewidth=1)
            ax.plot([period, period], [0, 1], ':k', linewidth=1)
        ax.axis([0, 30, 0, 1])

    axes[-1].set_xlabel('$t$')
    plt.tight_layout()
    plt.show()

In [ ]:
traces_none, traces_weak, traces_strong = simulate_lif_pulse_panels()
plot_lif_pulse_panels(traces_none, traces_weak, traces_strong)

In [ ]:
interact(lambda I=0.12: plot_lif_pulse_panels(*simulate_lif_pulse_panels(I=I)),
         I=(0.11, 0.25, 0.005));

## RTM Limit Cycle and Pulsed Simulation (shared by the RTM examples below)

`rtm_init` finds the single-cell RTM limit cycle at a given drive (plain
Heun integration until the 5th spike) and interpolates $(v,h,n)$ at each
requested phase, exactly as in Chapters 24-25 -- this is how the RTM
examples below splay-initialize their population. `simulate_rtm` then runs
$N$ such splay-initialized RTM cells, all receiving a common decaying
inhibitory conductance $g_{\rm syn}e^{-t/\tau_{\rm syn}}(v_{\rm syn}-v)$
starting at $t=0$, and returns the mean first-spike time ("period") and,
optionally, the full voltage traces.

In [ ]:
def alpha_h(v):
    return 0.128 * exp(-(v + 50) / 18)


def alpha_m(v):
    return 0.32 * (v + 54) / (1 - exp(-(v + 54) / 4))


def alpha_n(v):
    return 0.032 * (v + 52) / (1 - exp(-(v + 52) / 5))


def beta_h(v):
    return 4. / (1 + exp(-(v + 27) / 5))


def beta_m(v):
    return 0.28 * (v + 27) / (exp((v + 27) / 5) - 1)


def beta_n(v):
    return 0.5 * exp(-(v + 57) / 40)


def m_inf(v):
    return alpha_m(v) / (alpha_m(v) + beta_m(v))


def h_inf(v):
    return alpha_h(v) / (alpha_h(v) + beta_h(v))


def n_inf(v):
    return alpha_n(v) / (alpha_n(v) + beta_n(v))


def rtm_init(i_ext, phi_vec, c=1., g_k=80., g_na=100., g_l=0.1, v_k=-100., v_na=50., v_l=-67.):
    '''find the RTM limit cycle at i_ext (plain-float Heun integration
    until the 5th spike), then interpolate (v, h, n) at phases phi_vec
    (fraction of the last full period, measured from the 4th spike).'''
    t_final = 5000.
    dt = 0.01
    dt05 = dt / 2

    v = [-70.]
    m = [m_inf(v[0])]
    h = [h_inf(v[0])]
    n = [n_inf(v[0])]
    t_spikes = []

    k = 0
    t = 0.
    while len(t_spikes) < 5 and t < t_final:
        vk, mk, hk, nk = v[k], m[k], h[k], n[k]
        v_inc = (g_k * nk ** 4 * (v_k - vk) + g_na * mk ** 3 * hk * (v_na - vk) + g_l * (v_l - vk) + i_ext) / c
        h_inc = alpha_h(vk) * (1 - hk) - beta_h(vk) * hk
        n_inc = alpha_n(vk) * (1 - nk) - beta_n(vk) * nk

        v_tmp = vk + dt05 * v_inc
        m_tmp = m_inf(v_tmp)
        h_tmp = hk + dt05 * h_inc
        n_tmp = nk + dt05 * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                  + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp

        v.append(vk + dt * v_inc)
        h.append(hk + dt * h_inc)
        n.append(nk + dt * n_inc)
        m.append(m_inf(v[-1]))

        if vk >= -20 and v[-1] < -20:
            t_spike = (k * dt * (-20 - v[-1]) + (k + 1) * dt * (20 + vk)) / (vk - v[-1])
            t_spikes.append(t_spike)
        t = (k + 1) * dt
        k += 1

    num = len(phi_vec)
    T = t_spikes[4] - t_spikes[3]
    out = np.zeros((num, 3))
    for i, phi0 in enumerate(phi_vec):
        t0 = phi0 * T + t_spikes[3]
        kk = int(t0 / dt)
        frac_hi = (t0 - kk * dt) / dt
        frac_lo = ((kk + 1) * dt - t0) / dt
        out[i, 0] = v[kk + 1] * frac_hi + v[kk] * frac_lo
        out[i, 1] = h[kk + 1] * frac_hi + h[kk] * frac_lo
        out[i, 2] = n[kk + 1] * frac_hi + n[kk] * frac_lo
    return out


def simulate_rtm(i_ext, g_syn, v_syn, tau_syn, N=10, t_final=30., dt=0.01, record_trace=False,
                  c=1., g_k=80., g_na=100., g_l=0.1, v_k=-100., v_na=50., v_l=-67.):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    phi_vec = np.arange(N - 1, -1, -1) / N + 1 / (2 * N)

    initial_vector = rtm_init(i_ext, phi_vec, c=c, g_k=g_k, g_na=g_na, g_l=g_l, v_k=v_k, v_na=v_na, v_l=v_l)
    v = initial_vector[:, 0].copy()
    m = m_inf(v)
    h = initial_vector[:, 1].copy()
    n = initial_vector[:, 2].copy()

    t_spikes = np.zeros(N)
    v_trace = np.zeros((N, m_steps + 1)) if record_trace else None
    if record_trace:
        v_trace[:, 0] = v

    for k in range(m_steps):
        t = k * dt
        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v)
                  + i_ext + g_syn * exp(-t / tau_syn) * (v_syn - v)) / c
        n_inc = alpha_n(v) * (1 - n) - beta_n(v) * n
        h_inc = alpha_h(v) * (1 - h) - beta_h(v) * h

        v_tmp = v + dt05 * v_inc
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc
        m_tmp = m_inf(v_tmp)

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                  + g_l * (v_l - v_tmp) + i_ext + g_syn * exp(-(t + dt05) / tau_syn) * (v_syn - v_tmp)) / c
        h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp

        v_old = v.copy()
        v = v + dt * v_inc
        h = h + dt * h_inc
        n = n + dt * n_inc
        m = m_inf(v)

        ind = np.where((v < -20) & (v_old >= -20))[0]
        if len(ind) > 0:
            t_old, t_new = k * dt, (k + 1) * dt
            t_spikes[ind] = (t_old * (-20 - v[ind]) + t_new * (v_old[ind] + 20)) / (v_old[ind] - v[ind])

        if record_trace:
            v_trace[:, k + 1] = v

    period = t_spikes.mean()
    return (period, v_trace) if record_trace else period

## RTM With Inhibitory Pulse

The conductance-based analogue of `LIF_WITH_INHIBITORY_PULSE`: $N=10$
splay-initialized RTM cells all receive a common decaying inhibitory pulse,
shown for no inhibition, a weak/slow synapse, and a strong/fast one.

In [ ]:
def simulate_rtm_pulse_panels(i_ext=1.2, v_syn=-75.):
    _, v_trace_none = simulate_rtm(i_ext, g_syn=0., v_syn=v_syn, tau_syn=1., record_trace=True)  # g_syn=0 -> no inhibition
    _, v_trace_weak = simulate_rtm(i_ext, g_syn=0.30, v_syn=v_syn, tau_syn=9., record_trace=True)
    _, v_trace_strong = simulate_rtm(i_ext, g_syn=2.25, v_syn=v_syn, tau_syn=1., record_trace=True)
    return v_trace_none, v_trace_weak, v_trace_strong


def plot_rtm_pulse_panels(v_trace_none, v_trace_weak, v_trace_strong, N=10, t_final=30., dt=0.01):
    t = np.arange(round(t_final / dt) + 1) * dt
    fig, axes = plt.subplots(3, 1, figsize=(8, 9))

    for ax, v_trace in zip(axes, (v_trace_none, v_trace_weak, v_trace_strong)):
        for k in range(N):
            ax.plot(t, v_trace[k], '-k', linewidth=1)
        ax.axis([0, t_final, -100, 50])
        ax.set_ylabel('$v$ [mV]')

    axes[-1].set_xlabel('$t$ [ms]')
    plt.tight_layout()
    plt.show()

In [ ]:
v_trace_none, v_trace_weak, v_trace_strong = simulate_rtm_pulse_panels()
plot_rtm_pulse_panels(v_trace_none, v_trace_weak, v_trace_strong)

In [ ]:
interact(lambda i_ext=1.2: plot_rtm_pulse_panels(*simulate_rtm_pulse_panels(i_ext=i_ext)),
         i_ext=(0.8, 2.0, 0.05));

## RTM Condition Numbers

The RTM analogue of `LIF_CONDITION_NUMBERS`: for a weak/slow and a
strong/fast synapse, `compute_rtm_condition_numbers` reports the baseline
mean first-spike time $P$ and the percent change in $P$ under a 1% change
in $i_{\rm ext}$, $g_{\rm syn}$, and $\tau_{\rm syn}$. Its three baseline
voltage traces are the same simulations as `RTM_WITH_INHIBITORY_PULSE`, so
`plot_rtm_pulse_panels` is reused to show them.

In [ ]:
def compute_rtm_condition_numbers(i_ext=1.2, v_syn=-75.):
    def trial(g_syn, tau_syn):
        P_base, v_trace = simulate_rtm(i_ext, g_syn, v_syn, tau_syn, record_trace=True)
        P_raised_tau_syn = simulate_rtm(i_ext, g_syn, v_syn, tau_syn * 1.01)
        P_raised_g_syn = simulate_rtm(i_ext, g_syn * 1.01, v_syn, tau_syn)
        P_lowered_I = simulate_rtm(i_ext * 0.99, g_syn, v_syn, tau_syn)
        pct = {
            'i_ext': (P_lowered_I - P_base) / P_base * 100,
            'g_syn': (P_raised_g_syn - P_base) / P_base * 100,
            'tau_syn': (P_raised_tau_syn - P_base) / P_base * 100,
        }
        return {'P_base': P_base, 'pct': pct, 'v_trace': v_trace}

    return {
        'none': trial(g_syn=0., tau_syn=1.),  # g_syn=0 -> no inhibition
        'weak': trial(g_syn=0.30, tau_syn=9.),
        'strong': trial(g_syn=2.25, tau_syn=1.),
    }

In [ ]:
results = compute_rtm_condition_numbers()
print("weak/slow synapse (g_syn=0.30, tau_syn=9):", results['weak']['P_base'], results['weak']['pct'])
print("strong/fast synapse (g_syn=2.25, tau_syn=1):", results['strong']['P_base'], results['strong']['pct'])

# this reproduces the same 3-panel rastergram as RTM_WITH_INHIBITORY_PULSE
# -- the book's reference figure for this example is effectively the same plot.
plot_rtm_pulse_panels(results['none']['v_trace'], results['weak']['v_trace'], results['strong']['v_trace'])

## River

A geometric picture of the same contraction: the theta-neuron phase
$\theta$ and a decaying inhibitory conductance $g_{\rm syn}$ form a flow in
the $(\theta,g_{\rm syn})$ plane. Trajectories converging toward the
separatrix (blue) illustrate stable synchrony; the unstable manifold (red)
organizes trajectories that diverge from it. The many black trajectories
and arrow placements were hand-tuned in the book's own MATLAB source purely
to make the figure legible -- there is no deeper system to the specific
points chosen.

In [ ]:
def rhs(theta, g_syn, tau_m=1., I=0.3, tau_I=9.):
    theta_inc = -np.cos(theta) / tau_m + (2 * I - g_syn) * (1 + np.cos(theta)) - g_syn * np.sin(theta)
    g_syn_inc = -g_syn / tau_I
    return theta_inc, g_syn_inc


def river_trajectory(theta0, g_syn0, forward=True, dt=0.01, tau_m=1., I=0.3, tau_I=9.):
    dt05 = dt / 2
    theta = [theta0]
    g_syn = [g_syn0]
    k = 0
    sign = 1 if forward else -1
    while -np.pi <= theta[k] <= np.pi:
        theta_inc, g_syn_inc = rhs(theta[k], g_syn[k], tau_m, I, tau_I)
        theta_tmp = theta[k] + sign * dt05 * theta_inc
        g_syn_tmp = g_syn[k] + sign * dt05 * g_syn_inc
        theta_inc, g_syn_inc = rhs(theta_tmp, g_syn_tmp, tau_m, I, tau_I)
        theta.append(theta[k] + sign * dt * theta_inc)
        g_syn.append(g_syn[k] + sign * dt * g_syn_inc)
        k += 1
    return np.array(theta), np.array(g_syn)


def simulate_river(M=6):
    trajectories = []
    for ijk in range(1, M + 1):
        trajectories.append(river_trajectory(-np.pi, ijk / M, forward=True))
    for ijk in range(1, M + 1):
        trajectories.append(river_trajectory(np.pi, ijk / M, forward=False))
    for ijk in range(1, M + 1):
        trajectories.append(river_trajectory(0., ijk / (M + 1), forward=True))
    for ijk in range(1, M + 1):
        trajectories.append(river_trajectory(0., ijk / (M + 1), forward=False))

    extra_trajectories = [
        river_trajectory(-2., 1., forward=True),
        river_trajectory(0., 1., forward=True),
        river_trajectory(np.pi, 0.075, forward=False),
    ]

    arrow_points = [
        (0., 4 / 7),
        (-2., 0.592),
        (-2., 0.145),
        (0.5, 0.203),
        (2.5, 0.18),
        (2.85, 0.52),
    ]

    theta_sep, g_syn_sep = river_trajectory(-1.23, 1., forward=True)
    g_ast = g_syn_sep[-1]
    arrow_sep = (-0.75, 0.25)

    theta_unstable, g_syn_unstable = river_trajectory(1.25, 0.25, forward=False)

    return {
        'trajectories': trajectories,
        'extra_trajectories': extra_trajectories,
        'arrow_points': arrow_points,
        'theta_sep': theta_sep, 'g_syn_sep': g_syn_sep, 'g_ast': g_ast, 'arrow_sep': arrow_sep,
        'theta_unstable': theta_unstable, 'g_syn_unstable': g_syn_unstable,
    }


def plot_river(river):
    fig, ax = plt.subplots(figsize=(7, 7))

    for theta, g_syn in river['trajectories'] + river['extra_trajectories']:
        ax.plot(theta, g_syn, '-k', linewidth=2)

    for theta0, g_syn0 in river['arrow_points']:
        v = rhs(theta0, g_syn0)
        draw_arrow(ax, (-np.pi, np.pi), (0, 1), theta0, g_syn0, v, epsilon=0.05, width=2, color='k')

    ax.plot(river['theta_sep'], river['g_syn_sep'], '-b', linewidth=4)
    v = rhs(*river['arrow_sep'])
    draw_arrow(ax, (-np.pi, np.pi), (0, 1), river['arrow_sep'][0], river['arrow_sep'][1], v,
               epsilon=0.05, width=4, color='b')

    ax.plot(river['theta_unstable'], river['g_syn_unstable'], '-r', linewidth=4)

    ax.plot(np.pi, river['g_ast'], '.k', markersize=15)
    ax.text(np.pi + 0.1, river['g_ast'], r'$g_\ast$', fontsize=20)

    ax.axis([-np.pi, np.pi, 0, 1])
    ax.set_box_aspect(1)
    ax.set_xlabel(r'$\theta$')
    ax.set_ylabel(r'$g_{\rm syn}$')
    plt.tight_layout()
    plt.show()

In [ ]:
river = simulate_river()
plot_river(river)

In [ ]:
interact(lambda M=6: plot_river(simulate_river(M=M)), M=(2, 12, 1));